# 06 · Training decoders
Train the hit/miss classifier and position/velocity regressors on saved activity.
Decoders are written to `decoders/<network_id>/`.

## Generate / load activity
Use `dataset.save_simulations` once, then reload the saved activity + positions.

In [ ]:
import torch
from flynet_robustness.models import get_flynet
from flynet_robustness.dataset import save_simulations, prepare_dataset
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model1, model2, idxs1, idxs2 = get_flynet('FlyNet', device=device)
# save_simulations(model1, idxs1, config_path='data/configs/train_configuration.pt',
#                  out_dir='simulations/FlyNet', device=device)
activity = torch.load('simulations/FlyNet/activity.pt', weights_only=False)['network_activity_all']

## Hit/miss decoder
Per-trial split, StandardScaler + tanh MLP (matches the paper); returns time-binned test accuracy.

In [ ]:
import numpy as np
from flynet_robustness.training import train_hit_decoder
cfg = torch.load('data/configs/train_configuration.pt', weights_only=False)
acc, test_acc = train_hit_decoder(activity, cfg['true_labels'], cfg['time_duration_conditions'],
                                  network_id='FlyNet', decoder='mlp', scale=True)
print('test accuracy', round(test_acc, 3))

## Position / velocity decoders

In [ ]:
from flynet_robustness.training import train_position_decoder
positions = torch.load('simulations/FlyNet/positions.pt', weights_only=False)['positions']
X, Y = prepare_dataset(activity, positions)
# train_position_decoder(X, Y, network_id='FlyNet', velocity=False, device=device)